# Deep Past Initiative – Submit Notebook (ByT5-base)

- Base notebook (train): `notebooks/002/[2]dpc-starter-train-v1.ipynb`
- Beam width: `num_beams=4`
- Preprocessing: matches the train notebook’s `normalize_transliteration()` + task prefixes

## Model path
`MODEL_DIR` is intentionally easy to swap. Attach your trained model as a Kaggle Dataset / Model, then set `MODEL_DIR` accordingly.


In [1]:
import os
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


'false'

In [2]:
@dataclass
class CFG:
    # Competition data path (Kaggle may mount it in either location)
    input_dirs: tuple[str, ...] = (
        "/kaggle/input/competitions/deep-past-initiative-machine-translation",
        "/kaggle/input/deep-past-initiative-machine-translation",
    )

    # >>> Replace this later (intentionally centralized)
    # Example (dataset): /kaggle/input/<your-dataset>/<your-model-folder>
    # Example (model):   /kaggle/input/models/<owner>/<model>/pytorch/<variant>/<version>
    model_dir: str = os.environ.get("MODEL_DIR", "/kaggle/input/notebooks/tatsuokoshida/2-dpc-starter-train-v2/byt5-akkadian-model/fulltrain_byt5-base_multitask")

    output_dir: str = "/kaggle/working"

    # Tokenization / generation (match train notebook defaults)
    max_input_length: int = 512
    max_new_tokens: int = 512

    # Inference
    batch_size: int = 8
    num_workers: int = 2
    num_beams: int = 4
    early_stopping: bool = True

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


cfg = CFG()
print("PyTorch:", torch.__version__)
print("Device :", cfg.device)


PyTorch: 2.9.0+cu126
Device : cuda


In [3]:
PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})


def normalize_transliteration(text: str) -> str:
    # Copied from `notebooks/002/[2]dpc-starter-train-v1.ipynb`
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # Unify common gap markers to a single token for better pattern consistency.
    text = text.replace("…", " <gap> ")
    text = re.sub(r"\b[xX]{1,}\b", " <gap> ", text)

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def resolve_input_dir(input_dirs: tuple[str, ...]) -> str:
    for d in input_dirs:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Competition input directory not found. Tried: " + ", ".join(input_dirs)
    )


def build_inputs(test_df: pd.DataFrame) -> tuple[list[str], list[str]]:
    if "id" not in test_df.columns:
        raise ValueError(f"test.csv must contain 'id'. columns={list(test_df.columns)}")
    ids = test_df["id"].astype(str).tolist()

    if "task" in test_df.columns:
        tasks = test_df["task"].astype(str).tolist()
    else:
        tasks = ["akk2en"] * len(test_df)

    if "source" in test_df.columns:
        sources = test_df["source"].astype(str).tolist()
    elif "transliteration" in test_df.columns:
        sources = test_df["transliteration"].astype(str).tolist()
    else:
        raise ValueError(
            "test.csv must contain either 'source' or 'transliteration'. "
            f"columns={list(test_df.columns)}"
        )

    inputs: list[str] = []
    for t, s in zip(tasks, sources):
        if t == "akk2en":
            inputs.append(PREFIX_AKK_EN + normalize_transliteration(s))
        elif t == "en2akk":
            inputs.append(PREFIX_EN_AKK + str(s))
        else:
            raise ValueError(f"Unknown task: {t}")

    return ids, inputs


In [4]:
def resolve_model_dir(model_dir: str) -> str:
    candidates = [
        model_dir,
        # Useful when you ran training in the same notebook session and kept outputs in /kaggle/working
        "/kaggle/working/byt5-akkadian-model/fulltrain_byt5-base_multitask",
    ]
    for d in candidates:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Model directory not found. Set MODEL_DIR to a mounted path containing the saved model. "
        f"Tried: {candidates}"
    )


input_dir = resolve_input_dir(cfg.input_dirs)
test_path = f"{input_dir}/test.csv"

print("Test:", test_path)

test_df = pd.read_csv(test_path)
print("Test rows:", len(test_df))
print("Columns:", list(test_df.columns))

model_dir = resolve_model_dir(cfg.model_dir)
print("Model:", model_dir)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to(cfg.device)
model.eval()

ids, inputs = build_inputs(test_df)
print("Prepared inputs:", len(inputs))


Test: /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
Test rows: 4
Columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
Model: /kaggle/input/notebooks/tatsuokoshida/2-dpc-starter-train-v2/byt5-akkadian-model/fulltrain_byt5-base_multitask


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Prepared inputs: 4


In [5]:
class TextDataset(Dataset):
    def __init__(self, texts: list[str]):
        self.texts = texts

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> str:
        return self.texts[idx]


def collate_fn(batch_texts: list[str]):
    return tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.max_input_length,
    )


ds = TextDataset(inputs)
dl = DataLoader(
    ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=(cfg.device.type == "cuda"),
)


def _move_to_device(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


def postprocess_text(text: str) -> str:
    # Train notebook did not apply special postprocessing; keep minimal + stable.
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


preds: list[str] = []
with torch.inference_mode():
    for batch in tqdm(dl, desc="Generating"):
        batch = _move_to_device(batch, cfg.device)
        gen_ids = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch.get("attention_mask"),
            num_beams=cfg.num_beams,
            early_stopping=cfg.early_stopping,
            max_new_tokens=cfg.max_new_tokens,
        )
        out_texts = tokenizer.batch_decode(
            gen_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        preds.extend([postprocess_text(t) for t in out_texts])

assert len(preds) == len(ids)
print("Preds:", len(preds))
print("Example:", preds[0][:200] if preds else "<empty>")


Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Preds: 4
Example: Thus Kanesh colony, say to the fees, our messenger, every single, and the trading stations: The tablet came to the City.""


In [6]:
sub_df = pd.DataFrame({"id": ids, "translation": preds})
sub_path = Path(cfg.output_dir) / "submission.csv"
sub_df.to_csv(sub_path, index=False)

print("Submission preview:")
display(sub_df.head())
print("Saved to:", str(sub_path))


Submission preview:


,id,translation
0,0,"Thus Kanesh colony, say to the fees, our messe..."
1,1,When a tablet comes up from the City reached a...
2,2,"Since you hear our letter, either he gave it t..."
3,3,"I sent our tablets to every single and the, to..."


Saved to: /kaggle/working/submission.csv
